In [1]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import asyncio
import uvicorn
import websockets
import json
from datetime import datetime, timezone

app = FastAPI()

vessels = {}
history = {}
MAX_HISTORY = 30


In [2]:
import os
from os import getenv
from dotenv import load_dotenv

API_KEY = os.getenv("AISSTREAM_API_KEY")


# print(f"aici am cheia: {API_KEY}")


In [3]:
@app.get("/vessels")
def get_vessels():
    return list(vessels.values())

@app.get("/trails")
def get_trails():
    return history

In [4]:
async def connect_ais_stream():

    async with websockets.connect("wss://stream.aisstream.io/v0/stream") as websocket:

        subscribe_message = {
            "APIKey": API_KEY,
            "BoundingBoxes": [
                [[42.827639, 25.718994],
                 [45.970243, 33.711548]]
            ],
            "FilterMessageTypes": ["PositionReport"]
        }

        await websocket.send(json.dumps(subscribe_message))

        async for message_json in websocket:
            message = json.loads(message_json)

            if message.get("MessageType") != "PositionReport":
                continue

            ais = message["Message"]["PositionReport"]
            mmsi = ais["UserID"]

            point = {
                "mmsi": mmsi,
                "lat": ais["Latitude"],
                "lon": ais["Longitude"],
                "course": ais.get("Cog", 0),
                "speed": ais.get("Sog", 0),
                "timestamp": datetime.now(timezone.utc).timestamp()
            }

            vessels[mmsi] = point

            if mmsi not in history:
                history[mmsi] = []

            history[mmsi].append([point["lat"], point["lon"]])

            if len(history[mmsi]) > MAX_HISTORY:
                history[mmsi].pop(0)

In [5]:
from fastapi.responses import HTMLResponse

@app.get("/", response_class=HTMLResponse)
def map_ui():
    return """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8"/>

    <link rel="stylesheet" href="https://unpkg.com/leaflet/dist/leaflet.css"/>
    <script src="https://unpkg.com/leaflet/dist/leaflet.js"></script>

    <style>
        .ship-icon {
            font-size: 18px;
            text-align: center;
            transition: transform 0.2s linear;
        }
    </style>
</head>

<body>
<div id="map" style="height: 100vh;"></div>

<script>

// 🗺️ MAP INIT (Black Sea)
const map = L.map('map').fitBounds([
    [42.827639, 25.718994],
    [45.970243, 33.711548]
]);

L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
    maxZoom: 18
}).addTo(map);

// 🚢 STATE
let markers = {};

// 🧭 ARROW ICON (rotation = course)
function shipIcon(course) {
    return L.divIcon({
        className: "ship-icon",
        html: `<div style="transform: rotate(${course || 0}deg)">🚢</div>`,
        iconSize: [20, 20]
    });
}

// 🧾 POPUP (ONLY MMSI AS REQUESTED)
function popup(v) {
    return `<b>MMSI:</b> ${v.mmsi}`;
}

// 🔄 UPDATE LOOP
async function update() {

    const res = await fetch('/vessels');
    const data = await res.json();

    data.forEach(v => {

        if (!markers[v.mmsi]) {

            markers[v.mmsi] = L.marker([v.lat, v.lon], {
                icon: shipIcon(v.course)
            })
            .addTo(map)
            .bindPopup(popup(v));

        } else {

            markers[v.mmsi]
                .setLatLng([v.lat, v.lon])
                .setIcon(shipIcon(v.course))
                .setPopupContent(popup(v));
        }
    });
}

setInterval(update, 2000);

</script>

</body>
</html>
"""

In [ ]:
loop = asyncio.get_event_loop()

# AIS stream
loop.create_task(connect_ais_stream())

# FastAPI server
config = uvicorn.Config(app, host="127.0.0.1", port=8000, loop="asyncio")
server = uvicorn.Server(config)

loop.create_task(server.serve())

<Task pending name='Task-44' coro=<Server.serve() running at /Users/mkl/Documents/seantinel/.venv/lib/python3.13/site-packages/uvicorn/server.py:77>>

INFO:     Started server process [21714]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:62758 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /trails HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO:     127.0.0.1:62758 - "GET /vessels HTTP/1.1" 200 OK
INFO: